# Notebook 01: Plant Disease Surveillance Data Acquisition & Exploratory Analysis
**Course:** CSET 369 — Time Series Analysis & Forecasting  
**Team:** Kartik Lamba & Suhani Vij  
**Dataset:** 1,320 monthly records across 10 crop pathogens (Jan 2015 – Dec 2025)

---
### Objectives:
1. Ingest the authoritative master epidemiological time series (`data/disease_timeseries_master.csv`).
2. Verify provenance, schema validation, and zero-inflation profiling across all 10 pathogens.
3. Perform STL seasonal decomposition and correlogram analysis (ACF/PACF).
4. Conduct formal stationarity diagnostics using Augmented Dickey-Fuller (ADF) and KPSS tests.
5. Compute environmental cross-correlations with climate covariates (temperature, humidity, rainfall).


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Plot styling
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["font.size"] = 11

print("Environment configured successfully.")


## 1. Data Ingestion & Dataset Provenance
We load `data/disease_timeseries_master.csv`, which combines GBIF open surveillance records with NASA POWER agroclimatological indicators.


In [ ]:
data_path = Path("../data/disease_timeseries_master.csv")
if not data_path.exists():
    data_path = Path("data/disease_timeseries_master.csv")

df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"])

print(f"Total Records: {len(df):,}")
print(f"Date Range:    {df['date'].min().strftime('%b %Y')} to {df['date'].max().strftime('%b %Y')} ({df['date'].nunique()} months)")
print(f"Pathogens:     {df['pathogen_slug'].nunique()} distinct species")
print(f"Features:      {df.shape[1]} columns")
df.head(5)[["date", "pathogen_slug", "common_name", "host_crop", "documented_occurrence_count", "temp_mean_c", "relative_humidity_pct"]]


## 2. Pathogen Profiling & Zero-Inflation Analysis
Plant disease time series feature high sparsity because pathogens remain dormant during dry/unfavorable seasons.


In [ ]:
summary = []
for slug, grp in df.groupby("pathogen_slug"):
    counts = grp["documented_occurrence_count"]
    zero_pct = (counts == 0).mean() * 100
    summary.append({
        "Pathogen Slug": slug,
        "Common Name": grp["common_name"].iloc[0],
        "Host Crop": grp["host_crop"].iloc[0],
        "Mean Count": round(counts.mean(), 2),
        "Max Count": counts.max(),
        "Zero Months (%)": round(zero_pct, 1),
    })

summary_df = pd.DataFrame(summary).sort_values("Zero Months (%)", ascending=False)
print(f"Overall Dataset Zero Months: {(df['documented_occurrence_count'] == 0).mean()*100:.1f}%")
summary_df


In [ ]:
# Visualize occurrence counts over time for sample pathogens
fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True)
sample_slugs = ["pyricularia_oryzae", "alternaria_solani", "ralstonia_solanacearum"]
colors = ["#2D6A4F", "#D97706", "#2563EB"]

for ax, slug, color in zip(axes, sample_slugs, colors):
    sub = df[df["pathogen_slug"] == slug]
    ax.plot(sub["date"], sub["documented_occurrence_count"], marker="o", markersize=3, color=color, label=sub["common_name"].iloc[0])
    ax.set_ylabel("Monthly Occurrences")
    ax.set_title(f"{sub['common_name'].iloc[0]} ({sub['host_crop'].iloc[0]})", fontweight="bold")
    ax.legend(loc="upper left")

plt.xlabel("Observation Date")
plt.tight_layout()
plt.show()


## 3. Seasonality & Time Series Decomposition
We perform classical seasonal decomposition on Rice Blast (*Pyricularia oryzae*) to isolate Trend, Seasonal, and Residual components.


In [ ]:
rice_blast = df[df["pathogen_slug"] == "pyricularia_oryzae"].set_index("date")["documented_occurrence_count"]
rice_blast_monthly = rice_blast.asfreq("MS").fillna(0)

decomp = seasonal_decompose(rice_blast_monthly, model="additive", period=12)

fig = decomp.plot()
fig.set_size_inches(14, 8)
plt.suptitle("Additive Decomposition (Period=12): Rice Blast Occurrences", y=1.02, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Formal Stationarity Testing (ADF & KPSS)
Stationarity is tested using both:
- **Augmented Dickey-Fuller (ADF)**: $H_0$ is unit root (non-stationary).
- **KPSS**: $H_0$ is trend-stationary.


In [ ]:
def run_stationarity_tests(series, name):
    adf_res = adfuller(series.dropna(), autolag="AIC")
    kpss_res = kpss(series.dropna(), regression="c", nlags="auto")
    
    adf_stationary = adf_res[1] < 0.05
    kpss_stationary = kpss_res[1] >= 0.05
    
    return {
        "Series": name,
        "ADF Stat": round(adf_res[0], 3),
        "ADF p-value": round(adf_res[1], 4),
        "ADF Result": "Stationary" if adf_stationary else "Non-Stationary",
        "KPSS Stat": round(kpss_res[0], 3),
        "KPSS p-value": round(kpss_res[1], 4),
        "KPSS Result": "Stationary" if kpss_stationary else "Non-Stationary",
    }

stat_rows = []
for slug in df["pathogen_slug"].unique():
    ser = df[df["pathogen_slug"] == slug]["documented_occurrence_count"]
    cname = df[df["pathogen_slug"] == slug]["common_name"].iloc[0]
    stat_rows.append(run_stationarity_tests(ser, cname))

pd.DataFrame(stat_rows)


## 5. Autocorrelation (ACF & PACF) Correlograms


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
plot_acf(rice_blast_monthly, lags=24, ax=ax1, title="ACF: Rice Blast Occurrences")
plot_pacf(rice_blast_monthly, lags=24, ax=ax2, title="PACF: Rice Blast Occurrences", method="ywm")
plt.tight_layout()
plt.show()


## 6. Environmental Cross-Correlation (Climate Drivers)
We evaluate Pearson correlations between occurrence counts and environmental covariates.


In [ ]:
weather_cols = ["temp_mean_c", "temp_max_c", "temp_min_c", "relative_humidity_pct", "precipitation_monthly_mm"]
corr_matrix = df[["documented_occurrence_count"] + weather_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation Matrix: Disease Occurrences vs. Meteorological Drivers", fontweight="bold")
plt.tight_layout()
plt.show()


### Summary of Notebook 01 Findings:
1. **Severe Zero-Inflation**: Overall zero-month proportion is **71.8%**, requiring specialized hurdle/gating techniques.
2. **Distinct Seasonality**: Decomposition confirms clear annual seasonality (peaks coincide with monsoon/post-monsoon moisture).
3. **Stationarity**: Raw counts exhibit borderline non-stationarity with localized variance clustering, motivating differencing ($d=1$) and GARCH volatility models in Notebook 02.
